# CampusGPT: A RAG-Based Student Assistant for UET Mardan

CampusGPT is a Retrieval-Augmented Generation (RAG) chatbot that answers student questions about the University of Engineering & Technology (UET) Mardan, covering fees, hostels, admissions, examination rules, scholarships, and official contacts. Its knowledge base is built from official university documents.

> **Disclaimer:** This is an independent academic project. It is not affiliated with or endorsed by UET Mardan. Answers are generated from publicly available documents and may be outdated or incomplete, so please verify important details with the relevant university office.

## Overview

Rather than relying on the language model's own memory, CampusGPT retrieves the most relevant passages from the university documents and asks the model to answer using only that material. Every response lists the source documents it was drawn from.

## Architecture

1. **Ingestion:** PDFs and text files are read and split into overlapping chunks (about 400 words, 50-word overlap).
2. **Embedding:** Each chunk is converted into a dense vector with the `all-MiniLM-L6-v2` Sentence-Transformers model.
3. **Indexing:** The vectors are stored in a FAISS index for fast similarity search.
4. **Retrieval:** For each question, the 4 most similar chunks are retrieved.
5. **Generation:** The retrieved context and the question are sent to an LLM through the Groq API. The prompt restricts the model to the supplied context and tells it not to guess numbers or dates.
6. **Interface:** A Gradio chat UI presents the answer together with its sources.

## Tech Stack

| Component | Tool |
|---|---|
| Language / Environment | Python, Google Colab |
| Document parsing | pypdf |
| Embeddings | Sentence-Transformers (`all-MiniLM-L6-v2`) |
| Vector search | FAISS (CPU) |
| LLM | Groq API (chat completions endpoint) |
| User interface | Gradio |

## Key Features

- Answers grounded in official documents, with source attribution
- Bilingual: replies in English or Roman Urdu, matching the language of the question
- Safe fallback when the answer is not in the documents, instead of guessing
- Request timeout and retry handling for API calls
- Runs entirely on free tools

## Prerequisites

1. **Groq API key** (free): create one at https://console.groq.com/keys
2. **Colab Secret:** in Colab, open the key icon in the left sidebar, add a secret named `UETchatbot` with your key as the value, and turn on *Notebook access*.
3. **Dataset:** a set of PDF and/or `.txt` files containing the university information you want the assistant to know about.

## How to Run

Run the cells from top to bottom (Shift + Enter). Upload your dataset in Step 3 and launch the chat interface in Step 9.

## Security Note

The API key is read from Colab Secrets and is never stored in this notebook, so the notebook can be shared or published without exposing it. Never paste a key directly into a cell.


## Step 1: Install Dependencies

Installs the libraries used for embeddings, vector search, the user interface, and PDF parsing.


In [1]:
!pip install -q sentence-transformers faiss-cpu gradio google-generativeai pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpc-google-iam-v1 0.14.6 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompat

## Step 2: Mount Google Drive (Optional)

Run this cell if your dataset is stored in Google Drive. Otherwise, skip it and upload your files directly in Step 3.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Step 3: Upload the Dataset

Upload your PDF and `.txt` files. They are stored in a working folder named `uet_data`. Run the cell below and select your files in the dialog, or use the file browser in the left sidebar.


In [3]:
import os

DATA_FOLDER = '/content/drive/MyDrive/Colab Notebooks/UET chatbot'
os.makedirs(DATA_FOLDER, exist_ok=True)

from google.colab import files
print('Apni PDF/txt files select karo (multiple select kar sakte ho):')
uploaded = files.upload()

import shutil
for fname in uploaded.keys():
    shutil.move(fname, os.path.join(DATA_FOLDER, fname))

print('\n✅ Files uet_data folder mein daal di gayi:')
print(os.listdir(DATA_FOLDER))

Apni PDF/txt files select karo (multiple select kar sakte ho):


Saving aacedamic calender 2024-25.pdf to aacedamic calender 2024-25.pdf
Saving campusgpt_faq_dataset.txt to campusgpt_faq_dataset.txt
Saving center of ai.pdf to center of ai.pdf
Saving clubs and society.pdf to clubs and society.pdf
Saving Contacts.pdf to Contacts.pdf
Saving eligibility createria.pdf to eligibility createria.pdf
Saving exam rules.pdf to exam rules.pdf
Saving fee structure.pdf to fee structure.pdf
Saving hostel regulation.pdf to hostel regulation.pdf
Saving hostel.pdf to hostel.pdf
Saving PG-Academic-Calender-24-25.pdf to PG-Academic-Calender-24-25.pdf
Saving PG-fee structure.pdf to PG-fee structure.pdf
Saving PG-Prospectus-2025-Final.pdf to PG-Prospectus-2025-Final.pdf
Saving PG-Schoalarships.pdf to PG-Schoalarships.pdf
Saving Prospectus-2026-27.pdf to Prospectus-2026-27.pdf
Saving summer-ac-2024.pdf to summer-ac-2024.pdf

✅ Files uet_data folder mein daal di gayi:
['uet_chatabot_dataset.pdf', 'Prospectus-2026-27.pdf', 'PG-Prospectus-2025-Final.pdf', 'PG-Schoalarships.p

## Step 4: Configure the Groq API Key

Installs the Groq client and loads the API key from Colab Secrets (secret name: `UETchatbot`). Make sure *Notebook access* is enabled for the secret.


In [4]:
!pip install groq


In [20]:
from google.colab import userdata

groq_api_key = userdata.get('UETchatbot')

if not groq_api_key:
    raise ValueError(
        "Groq API key not found in Colab Secrets. "
        "Add a secret named 'UETchatbot' with your Groq API key, "
        "and make sure 'Notebook access' is toggled ON."
    )

print("Groq API key loaded from Colab Secrets.")

Groq API key loaded from Colab Secrets.


## Step 5: Load and Chunk the Documents

Reads every PDF and text file in the data folder and splits the text into overlapping chunks of about 400 words. Smaller chunks make retrieval more precise, and each chunk keeps track of its source file so answers can cite it.


In [21]:
from pypdf import PdfReader
import glob

def load_texts(folder):
    all_data = []
    for f in glob.glob(f"{folder}/*"):
        try:
            if f.endswith(".pdf"):
                reader = PdfReader(f)
                text = " ".join(page.extract_text() or "" for page in reader.pages)
            elif f.endswith(".txt"):
                text = open(f, encoding="utf-8", errors="ignore").read()
            else:
                continue
            all_data.append((os.path.basename(f), text))
        except Exception as e:
            print(f"Skip {f}: {e}")
    return all_data

def chunk_text(text, size=400, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), size - overlap):
        chunk = " ".join(words[i:i+size])
        if len(chunk.strip()) > 20:  # empty/junk chunks skip karo
            chunks.append(chunk)
    return chunks

docs = load_texts(DATA_FOLDER)

chunks, sources = [], []
for fname, text in docs:
    for c in chunk_text(text):
        chunks.append(c)
        sources.append(fname)

print(f"Total files loaded: {len(docs)}")
print(f"Total chunks created: {len(chunks)}")

Total files loaded: 17
Total chunks created: 415


## Step 6: Build the Embeddings and FAISS Index

Converts each chunk into a vector embedding with `all-MiniLM-L6-v2` and indexes the vectors in FAISS. This index serves as the chatbot's searchable knowledge base.


In [22]:
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(chunks, show_progress_bar=True)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

print("✅ Brain ready with", index.ntotal, "chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Brain ready with 415 chunks


## Step 7: Define the Retrieval and Generation Function

For each question, this function embeds the query, retrieves the 4 most relevant chunks, builds a prompt that limits the model to that context, and calls the Groq API. The response is returned with the names of the source documents. Timeout and retry handling are included.


In [23]:
import requests
import json
import numpy as np

def get_answer(query, k=4, retries=2):
    q_emb = embedder.encode([query])
    distances, idx = index.search(np.array(q_emb), k)

    # Clean flattening to prevent any internal vector index errors
    flat_idx = idx.flatten()
    context = "\n\n".join([chunks[int(i)] for i in flat_idx])
    used_sources = list(dict.fromkeys([sources[int(i)] for i in flat_idx]))

    prompt = f"""Tum CampusGPT ho, UET Mardan ka official AI assistant,
students ke liye bana hua.

Rules:
- Sirf neeche diye context se jawab do
- Agar context mein jawab nahi hai to saaf bolo: "Ye info abhi mere paas nahi hai, concerned office se confirm kar lein."
- Kabhi bhi guess/andaza mat lagao numbers ya dates ke baare mein
- Roman Urdu ya English mein jawab do, jismein sawal poocha gaya ho
- Jawab short aur clear rakho

Context:
{context}

Student ka sawal: {query}
Jawab:"""

    # 1. Sahi Base Endpoint
    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {groq_api_key}",
        "Content-Type": "application/json"
    }

    # 2. FIXED: Active model 'llama-3.3-70b-versatile' use kiya hai
    payload = {
        "model": "openai/gpt-oss-120b",
        "messages": [{"role": "user", "content": prompt}]
    }

    for attempt in range(retries):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=30)

            # Print explicit server responses if something drops
            if response.status_code != 200:
                print(f"❌ Groq Server Error {response.status_code}: {response.text}")
                continue

            res_data = response.json()
            # 3. FIXED: Added correct list index [0] to parse choices properly
            answer = res_data['choices'][0]['message']['content']
            return f"{answer}\n\n📌 Source: {', '.join(used_sources)}"

        except Exception as e:
            if attempt < retries - 1:
                print(f"⏳ Retry ho raha hai... ({attempt+1}/{retries})")
                continue
            return f"⚠️ System issue. (Error: {e})"

print('✅ Stable Groq function configured with Llama-3.3-70B!')


✅ Stable Groq function configured with Llama-3.3-70B!


In [24]:
import requests

k = groq_api_key
print("Length:", len(k))
print("Starts with:", k[:4])
print("Has extra spaces/newline:", k != k.strip())

r = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {k.strip()}"}
)
print("Groq says:", r.status_code)   # 200 = key valid, 401 = key invalid

Length: 56
Starts with: gsk_
Has extra spaces/newline: False
Groq says: 200


## Step 8: Test the Assistant

Run a few sample questions to check that the answers are accurate and the sources are shown. The assistant replies in the language of the question.


In [25]:
print(get_answer("Hostel gate kis time band hota hai?"))

Hostel gate har season (Winter Oct‑Mar aur Summer Apr‑Sep) 4:00 PM ko band hoti hai (gate ka opening time 7:45 AM hota hai).

📌 Source: campusgpt_faq_dataset.txt, Prospectus-2026-27.pdf, uet_chatabot_dataset.pdf


In [26]:
print(get_answer("BS AI ki fee kitni hai?"))

❌ Groq Server Error 429: {"error":{"message":"Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0hm33pqezkakh1s7b2m752v` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 2120, Requested 5956. Please try again in 570ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

BS Artificial Intelligence ki fee **Rs. 210,000** (self‑finance, per semester).

📌 Source: campusgpt_faq_dataset.txt


## Step 9: Launch the Chat Interface

Starts a Gradio chat interface with example questions. To create a temporary public link for sharing a demo, change `share=False` to `share=True` in the launch call.


In [28]:
import gradio as gr

def chat_fn(message, history):
    return get_answer(message)

demo = gr.ChatInterface(
    fn=chat_fn,
    title="🎓 CampusGPT - UET Mardan",
    description="Fee, hostel, admission, exam dates, scholarship — kuch bhi poocho!",
    examples=[
        "Hostel fee kitni hai?",
        "BS AI ki fee kitni hai?",
        "Attendance kitni chahiye exam ke liye?",
        "Registrar ka email kya hai?"
    ]
)

demo.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

---
## Limitations and Future Work

**Current limitations**
- Knowledge is limited to the documents provided. Topics such as library rules, transport schedules, the faculty directory, and exam datesheets are not yet covered.
- Fee figures and academic calendars change over time, so the source documents must be kept up to date.

**Planned improvements**
- Expand the dataset, and collect missing information from students and university offices (for example, through a survey form)
- Build an evaluation set of 20 to 30 test questions to measure answer accuracy and improve the data where answers fall short
- Deploy permanently on Hugging Face Spaces as an `app.py`
- Add a feedback option ("Was this helpful?") to guide improvements
